# Day 39 — **Episodic Memory: Python Implementation**

**Phase 2 · Module 6: Agent Memory, Tools & MCP** · ~30 min

Yesterday's module notebook named the four memory types; today we *build* the episodic one properly — the store that answers "what happened before with this customer?" It's an `EpisodicMemory` class with `store(event)` and `retrieve(query, top_k)`, backed by **FAISS** for similarity search and a **recency bias** so recent events win ties.

Real FAISS here (installed). To keep it offline we use a small **deterministic hashing embedder** instead of downloading `sentence-transformers` — the retrieval logic is identical; only the vectoriser swaps. Install for real: `pip install faiss-cpu sentence-transformers`.

Today:
1. A deterministic **embedder** (swap-in point for sentence-transformers).
2. `EpisodicMemory` with a **FAISS** index — store + similarity retrieve.
3. **Recency bias** — blend semantic score with event age.
4. **Persistence** — serialise/deserialise memory to JSON.

## 1. A deterministic embedder

Retrieval needs vectors. In production that's `sentence-transformers`; here we hash tokens into a fixed-dim vector and L2-normalise, so cosine similarity = dot product. Deterministic, offline, same interface (`encode(texts) -> ndarray`).

In [1]:
import numpy as np, hashlib

DIM = 64

def _bucket(tok: str) -> int:
    # stable hash (Python's hash() is per-process randomised -> not reproducible)
    return int(hashlib.md5(tok.encode()).hexdigest(), 16) % DIM

def encode(texts: list[str]) -> np.ndarray:
    """Deterministic bag-of-hashed-tokens embedder. Swap for SentenceTransformer.encode."""
    vecs = np.zeros((len(texts), DIM), dtype="float32")
    for i, t in enumerate(texts):
        for tok in t.lower().split():
            vecs[i, _bucket(tok)] += 1.0               # hashing trick
    norms = np.linalg.norm(vecs, axis=1, keepdims=True)
    return vecs / np.clip(norms, 1e-9, None)          # L2-normalise -> cosine via dot

a, b = encode(["overdraft fee dispute", "overdraft fee refund"])
c, = encode(["mortgage application"])
print(f"related texts   cos={a @ b:.2f}")
print(f"unrelated texts cos={a @ c:.2f}")

related texts   cos=0.67
unrelated texts cos=0.00


☝ The only thing to hold onto: `encode` returns **L2-normalised** vectors, so a dot product *is* cosine similarity (Day 20) — which lets FAISS's inner-product index rank by semantic closeness. The hashing embedder is crude (no real semantics), but it proves the plumbing offline; swapping `SentenceTransformer("all-MiniLM-L6-v2").encode` in changes `DIM` to 384 and gives real meaning, with the rest of the class untouched.

## 2. EpisodicMemory over FAISS

`IndexFlatIP` does exact inner-product search. Store: embed the event, add to the index, keep the metadata alongside. Retrieve: embed the query, search top-k.

In [2]:
import faiss, time
from dataclasses import dataclass, field, asdict

@dataclass
class Event:
    customer_id: str
    text: str
    ts: float = field(default_factory=time.time)

class EpisodicMemory:
    def __init__(self, dim: int = DIM):
        self.index = faiss.IndexFlatIP(dim)           # exact cosine (vectors normalised)
        self.events: list[Event] = []

    def store(self, event: Event) -> None:
        self.index.add(encode([event.text]))
        self.events.append(event)

    def _search(self, query: str, k: int):
        if not self.events:
            return []
        scores, idxs = self.index.search(encode([query]), min(k, len(self.events)))
        return [(self.events[i], float(scores[0][j]))
                for j, i in enumerate(idxs[0]) if i != -1]

mem = EpisodicMemory()
DAY = 86400; now = time.time()
for cid, txt, age in [
    ("C-4417", "disputed a 48 GBP overdraft fee, refunded", 5),
    ("C-4417", "asked to increase overdraft limit", 200),
    ("C-4417", "routine balance check", 1),
    ("C-9001", "flagged for sanctions review", 2)]:
    mem.store(Event(cid, txt, now - age * DAY))

for ev, score in mem._search("overdraft fee complaint", k=3):
    print(f"  {score:.2f}  [{ev.customer_id}]  {ev.text}")

  0.52  [C-4417]  asked to increase overdraft limit
  0.48  [C-4417]  disputed a 48 GBP overdraft fee, refunded
  0.00  [C-4417]  routine balance check


☝ FAISS separates the **vectors** (in the index) from the **metadata** (the parallel `events` list) — they stay aligned because both append in lockstep, and `search` returns positional indices into `events`. Notice the flaw already: the top hit ignores *which customer* and *how old* the event is — pure semantic search will happily return C-9001's data for a C-4417 query. That's the two things we fix next: scope and recency.

## 3. Recency bias + customer scope

Retrieval quality = semantic relevance **and** recency **and** scope. Filter to the customer, then blend the FAISS score with an age-decay term so a fresh dispute outranks an old, equally-relevant note.

In [3]:
def retrieve(mem: EpisodicMemory, customer_id: str, query: str,
             top_k: int = 3, now: float | None = None, recency_weight: float = 0.3):
    now = now or time.time()
    hits = mem._search(query, k=len(mem.events))         # get all, then filter+rerank
    ranked = []
    for ev, sem in hits:
        if ev.customer_id != customer_id:                # SCOPE: this customer only
            continue
        age_days = (now - ev.ts) / DAY
        recency = 1 / (1 + age_days)                      # newer -> ~1, older -> ~0
        final = (1 - recency_weight) * sem + recency_weight * recency
        ranked.append((final, sem, recency, ev.text))
    ranked.sort(reverse=True)
    return ranked[:top_k]

print("C-4417 recall for 'overdraft problem' (scoped + recency-weighted):")
for final, sem, rec, text in retrieve(mem, "C-4417", "overdraft problem", now=now):
    print(f"  final={final:.2f} (sem={sem:.2f} rec={rec:.2f})  {text}")

C-4417 recall for 'overdraft problem' (scoped + recency-weighted):
  final=0.46 (sem=0.59 rec=0.17)  disputed a 48 GBP overdraft fee, refunded
  final=0.22 (sem=0.32 rec=0.00)  asked to increase overdraft limit
  final=0.15 (sem=0.00 rec=0.50)  routine balance check


☝ Two guards turn raw search into *useful* memory. **Scope** (`ev.customer_id != customer_id`) makes C-9001's sanctions flag unreachable for C-4417 — a hard compliance boundary, not a ranking preference. **Recency blend** (`(1-w)*sem + w*recency`) is a tunable dial: `recency_weight=0.3` keeps semantics dominant but breaks ties toward fresh events. Tune `w` per use case — a fraud agent wants recent-heavy, a policy-lookup agent wants semantics-only (`w=0`).

## 4. Persistence — memory that survives a restart

Working memory that dies with the process isn't memory. Serialise events to JSON on shutdown; on startup, reload and rebuild the FAISS index from the stored text.

In [4]:
import json

def save(mem: EpisodicMemory, path: str) -> None:
    with open(path, "w") as f:
        json.dump([asdict(e) for e in mem.events], f)

def load(path: str) -> EpisodicMemory:
    m = EpisodicMemory()
    with open(path) as f:
        for row in json.load(f):
            m.store(Event(**row))                         # re-embeds + re-indexes
    return m

import tempfile, os
path = os.path.join(tempfile.gettempdir(), "episodic_C-4417.json")
save(mem, path)
reloaded = load(path)
print("reloaded events:", len(reloaded.events))
top = retrieve(reloaded, "C-4417", "overdraft problem", now=now)[0]
print("top after reload:", top[3])

reloaded events: 4
top after reload: disputed a 48 GBP overdraft fee, refunded


☝ We persist the **events (text + metadata)**, not the FAISS vectors, and rebuild the index on load — cheaper to store, and it re-embeds with whatever the current embedder is (so upgrading the model just means a reload). For scale this JSON file becomes a real store: **pgvector** (tomorrow, Day 40) or a managed vector DB holds both vectors and metadata durably, and FAISS is swapped for the DB's ANN index. The `store`/`retrieve` interface — the part your agents depend on — stays exactly the same.

## Recap

| Piece | Implementation |
|---|---|
| Vectors | L2-normalised embeddings (`encode`); cosine = dot |
| Similarity search | `faiss.IndexFlatIP` + parallel metadata list |
| Scope | filter by `customer_id` — compliance boundary |
| Recency | blend `sem` with `1/(1+age_days)`, tunable weight |
| Persistence | JSON events + rebuild index on load |

**Swap to prod:** hashing embedder → `sentence-transformers`; FAISS/JSON → pgvector. Tomorrow (Day 40): semantic memory on a real vector store — pgvector with `asyncpg`.